In [6]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [7]:
X,y = make_classification(n_samples=10000,n_features=10,n_informative=3)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.889


## Bagging

In [16]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9325


In [17]:
bag.estimators_samples_[0].shape

(4000,)

In [18]:
bag.estimators_features_[0].shape

(10,)

## Bagging using svm

In [20]:
bag = BaggingClassifier(
    estimator = SVC(),
    n_estimators=500,
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

0.918
(4000,)
(10,)


## Pasting

In [21]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose=True,
    n_jobs=-1
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    8.5s remaining:  1.0min
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    9.3s finished
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    0.0s remaining:    0.6s
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    0.2s finished


0.9325
(2000,)
(10,)


## Random Subspace

In [22]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    random_state=42,
    max_features=0.5,
    bootstrap_features=True,
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

0.9245
(8000,)
(5,)


## random Patches

In [23]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42,
    max_features=0.5,
    bootstrap_features=True,
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

0.9215
(2000,)
(5,)


## OOB score

In [24]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42,
    oob_score=True
)

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print(accuracy_score(y_test,y_pred))
print(bag.estimators_samples_[0].shape)
print(bag.estimators_features_[0].shape)

0.9325
(2000,)
(10,)


## Bagging Tips
* Bagging generally gives better results than Pasting
* Good results come around the 25% to 50% row sampling mark
* Random patches and subspaces should be used while dealing with high dimensional data
* To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

## GridSearchCV

In [28]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'n_estimators':[5,10,50],
    'max_samples':[0.1,0.4,0.7,1.0],
    'bootstrap':[True,False],
    'max_features':[0.1,0.4,0.7,1.0]
}

search = GridSearchCV(BaggingClassifier(),parameters,cv=5,n_jobs=-1)
search.fit(X_train,y_train)
print(search.best_params_)
print(search.best_score_)


{'bootstrap': False, 'max_features': 0.7, 'max_samples': 0.7, 'n_estimators': 50}
0.927125
